# Getting Started with Gemini API: A Comprehensive Practical Guide

Current Google GenAI SDK examples for text, images, conversation and embeddings. Live requests default OFF. Set GEMINI_API_KEY and TDB_RUN_GEMINI=1 explicitly to call Google. No live Gemini calls were run for this import because no key was available. Offline transport tests below are synthetic SDK contract checks only, not inference validation. Historical source transcripts remain in the appendix.

Run cells in order. Dependencies and hardware requirements are listed below.

In [ ]:
import os, io, hashlib
from pathlib import Path
from urllib.request import urlopen, Request
from PIL import Image
from google import genai
from google.genai import types

# Install: python -m pip install google-genai==2.34.0 Pillow==12.1.1 httpx==0.28.1
# Enable explicitly only after setting GEMINI_API_KEY and reviewing provider pricing.
RUN_LIVE = os.environ.get('TDB_RUN_GEMINI') == '1'
if RUN_LIVE and not os.environ.get('GEMINI_API_KEY'):
    raise RuntimeError('Set GEMINI_API_KEY in your local environment; never put it in a notebook.')
client = genai.Client(http_options=types.HttpOptions(timeout=60000)) if RUN_LIVE else None
MODEL = os.environ.get('GEMINI_MODEL', 'gemini-3.8-flash')
EMBEDDING_MODEL = os.environ.get('GEMINI_EMBEDDING_MODEL', 'gemini-embedding-001')
def response_text(response):
    if not response.candidates or not response.text:
        raise RuntimeError('No text candidate returned; inspect safety/finish information.')
    return response.text
print('Live Gemini requests enabled:', RUN_LIVE)

In [ ]:
def generate_text(api):
    response = api.models.generate_content(model=MODEL, contents='What is Large Language Model?')
    print(response_text(response))
    print('Candidates:', len(response.candidates))
    return response
if client is not None:
    text_response = generate_text(client)

In [ ]:
generation_config = types.GenerateContentConfig(candidate_count=1, stop_sequences=['OpenAI'],
    max_output_tokens=200, temperature=0.7)
def configured_text(api):
    response = api.models.generate_content(model=MODEL,
        contents='Tell me the story of the rise of LLM and chatbots.', config=generation_config)
    print(response_text(response))
    return response
if client is not None:
    configured_response = configured_text(client)

In [ ]:
# The original article's public sample, not a missing /content/images.jpg file.
IMAGE_URL = 'https://cdn-images-1.medium.com/max/800/1*KCm67shVi3tlAwJAzLZRqQ.jpeg'
with urlopen(Request(IMAGE_URL, headers={'User-Agent':'Mozilla/5.0'}), timeout=60) as response:
    image_bytes = response.read(10_000_001)
if len(image_bytes) > 10_000_000:
    raise ValueError('Image exceeds the example size bound.')
image = Image.open(io.BytesIO(image_bytes))
image.load()
if image.format not in ('JPEG', 'PNG', 'WEBP'):
    raise ValueError('Unsupported sample image format.')
image_part = types.Part.from_bytes(data=image_bytes, mime_type=Image.MIME[image.format])
print('Sample image:', image.size, image.format)
print('SHA256:', hashlib.sha256(image_bytes).hexdigest())

In [ ]:
def understand_image(api):
    response = api.models.generate_content(model=MODEL, contents=[image_part])
    print(response_text(response))
    return response
if client is not None:
    vision_response = understand_image(client)

In [ ]:
def multimodal_post(api):
    response = api.models.generate_content(model=MODEL, contents=[
        'Write a blog post about palestine right using the image as reference.', image_part])
    print(response_text(response))
    return response
if client is not None:
    post_response = multimodal_post(client)

In [ ]:
def converse(api):
    chat = api.chats.create(model=MODEL, history=[])
    first = chat.send_message('Can you please guide me on how to start with Large Langeuag Models?')
    print(response_text(first))
    second = chat.send_message('How can I fine-tune Large Language Models?')
    print(response_text(second))
    history = chat.get_history()
    assert [turn.role for turn in history] == ['user', 'model', 'user', 'model']
    for turn in history:
        print(turn.role, ':', ' '.join(part.text or '' for part in turn.parts))
    return history
if client is not None:
    chat_history = converse(client)

In [ ]:
def embed_text(api, texts):
    response = api.models.embed_content(model=EMBEDDING_MODEL, contents=texts,
        config=types.EmbedContentConfig(task_type='RETRIEVAL_DOCUMENT', title='Embedding of Dota 2 question'))
    vectors = [embedding.values for embedding in response.embeddings]
    assert len(vectors) == (1 if isinstance(texts, str) else len(texts))
    import math
    assert all(vector and all(math.isfinite(x) for x in vector) for vector in vectors)
    assert len({len(vector) for vector in vectors}) == 1
    print([vector[:10] for vector in vectors])
    return vectors
if client is not None:
    single_vectors = embed_text(client, 'Can you please guide me on how to start playing Dota 2?')
    batch_vectors = embed_text(client, ['Can you please guide me on how to start playing Dota 2?', 'Which Dota 2 heroes should I start with?'])

In [ ]:
# Offline SDK contract checks ONLY. This transport never reaches Google.
# Synthetic responses verify serialization/parsing, not model behavior or API acceptance.
import json, httpx, contextlib
requests = []
def offline_transport(request):
    payload = json.loads(request.content)
    requests.append((request.url.path, payload))
    if request.url.path.endswith(':embedContent'):
        return httpx.Response(200, json={'embedding': {'values': [0.25, -0.5, 0.75]}})
    if request.url.path.endswith(':batchEmbedContents'):
        return httpx.Response(200, json={'embeddings': [
            {'values': [0.25, -0.5, 0.75]} for _ in payload['requests']]})
    return httpx.Response(200, json={'candidates': [{'content': {'role': 'model',
        'parts': [{'text': 'OFFLINE TRANSPORT FIXTURE — not Gemini-generated.'}]}, 'finishReason': 'STOP'}]})
transport = httpx.MockTransport(offline_transport)
offline = genai.Client(api_key='offline-placeholder-not-a-secret',
    http_options=types.HttpOptions(client_args={'transport': transport}))
with contextlib.redirect_stdout(io.StringIO()):
    generate_text(offline); configured_text(offline); understand_image(offline); multimodal_post(offline)
    converse(offline)
    embed_text(offline, 'Can you please guide me on how to start playing Dota 2?')
    embed_text(offline, ['Can you please guide me on how to start playing Dota 2?', 'Which Dota 2 heroes should I start with?'])
offline.close()
generated = [p for path,p in requests if path.endswith(':generateContent')]
assert len(generated) == 6
assert generated[0]['contents'][0]['parts'][0]['text'] == 'What is Large Language Model?'
assert generated[1]['generationConfig']['candidateCount'] == 1
assert generated[1]['generationConfig']['stopSequences'] == ['OpenAI']
inline = generated[2]['contents'][0]['parts'][0]['inlineData']
assert inline['mime_type'] == Image.MIME[image.format]
import base64
assert base64.urlsafe_b64decode(inline['data']) == image_bytes
assert len(generated[-1]['contents']) == 3  # first user/model turns plus the follow-up
assert any(path.endswith(':batchEmbedContents') for path,p in requests)
assert types.GenerateContentConfig(**generation_config.model_dump()).max_output_tokens == 200
try:
    response_text(types.GenerateContentResponse(candidates=[]))
except RuntimeError:
    pass
else:
    raise AssertionError('Empty candidate response accepted')
if client is not None:
    client.close()
print('Offline SDK request/response contract checks passed; no Gemini inference is validated by these fixtures.')

## Original archive appendix — not executable

These verbatim source blocks and transcripts preserve the historical examples. They are not the maintained workflow above.

### Original code 1

```text
!pip install -q -U google-generativeai
```

### Original code 2

```text
import google.generativeai as genai
```

### Original code 3

```text
from IPython.display import Markdown

model = genai.GenerativeModel('gemini-pro')
response = model.generate_content("What is Large Language Model?")

Markdown(response.text)
```

### Original code 4

```text
response = model.generate_content(
    'Tell me the story of the rise of LLM and chatbots.',
    generation_config=genai.types.GenerationConfig(
        candidate_count=1,
        stop_sequences=['OpenAI'],
        max_output_tokens=200,
        temperature=0.7)
)

Markdown(response.text)
```

### Original code 5

```text
import PIL.Image

img = PIL.Image.open('/content/images.jpg')

img
```

### Original code 6

```text
model = genai.GenerativeModel('gemini-pro-vision')

response = model.generate_content(img)

Markdown(response.text)
```

### Original code 7

```text
response = model.generate_content(["Write a blog post about palestine right using the image as reference.", img])

Markdown(response.text)
```

### Original code 8

```text
model = genai.GenerativeModel('gemini-pro')

chat = model.start_chat(history=[])

chat.send_message("Can you please guide me on how to start with Large Langeuag Models?")

chat.history
```

### Original code 9

```text
output = genai.embed_content(
    model="models/embedding-001",
    content="Can you please guide me on how to start playing Dota 2?",
    task_type="retrieval_document",
    title="Embedding of Dota 2 question")

print(output['embedding'][0:10])
```

### Original code 10

```text
output = genai.embed_content(
    model="models/embedding-001",
    content=[
        "Can you please guide me on how to start playing Dota 2?",
        "Which Dota 2 heroes should I start with?",
    ],
    task_type="retrieval_document",
    title="Embedding of Dota 2 question")

for emb in output['embedding']:
    print(emb[:10])
```

### Original transcript 1

The Temple Mount is a holy site located in Jerusalem. It is the holiest site in Judaism and the third holiest site in Islam. The Temple Mount is home to the Al-Aqsa Mosque, the Dome of the Rock, and the Western Wall. The Temple Mount is a disputed territory, and its status is a major source of conflict between Israelis and Palestinians.

### Original transcript 2

The image shows two young men standing in front of the Dome of the Rock, a Muslim shrine in Jerusalem. They are both wearing Palestinian flags around their shoulders. The image is a powerful symbol of Palestinian identity and resistance. It is a reminder of the Palestinian people’s long struggle for freedom and independence.

### Original transcript 3

The Dome of the Rock is a holy site for Muslims, Jews, and Christians. It is located in the Old City of Jerusalem, which is a disputed territory between Israel and Palestine. The Palestinians believe that the Dome of the Rock is the site of the Prophet Muhammad’s ascent to heaven. The Israelis believe that the Dome of the Rock is the site of the Second Temple.

### Original transcript 4

The Dome of the Rock has been a flashpoint of violence between Israelis and Palestinians for many years. In 1967, Israel captured the Old City of Jerusalem from Jordan. Since then, the Israelis have controlled the Dome of the Rock. However, the Palestinians still consider the Dome of the Rock to be a Muslim holy site.

### Original transcript 5

In 2000, the Israeli Prime Minister Ariel Sharon visited the Dome of the Rock. This visit sparked the Second Intifada, a Palestinian uprising that lasted for five years. The Second Intifada ended in 2005, but the conflict between Israelis and Palestinians continues.

### Original transcript 6

The Dome of the Rock is a symbol of the Palestinian people’s struggle for freedom and independence. It is a reminder of the long history of conflict between Israelis and Palestinians. The Dome of the Rock is also a symbol of hope for peace. It is a reminder that Israelis and Palestinians must find a way to live together in peace.

### Original transcript 7

[0.060604308, -0.023885584, -0.007826327, -0.070592545, 0.021225851, 0.043229062, 0.06876691, 0.049298503, 0.039964676, 0.08291664]

### Original transcript 8

[0.060604308, -0.023885584, -0.007826327, -0.070592545, 0.021225851, 0.043229062, 0.06876691, 0.049298503, 0.039964676, 0.08291664]

### Original transcript 9

[0.04775657, -0.044990525, -0.014886052, -0.08473655, 0.04060122, 0.035374347, 0.031866882, 0.071754575, 0.042207796, 0.04577447]